In [1]:
# System Imports
import sys
sys.path.insert(0, "../src")
import codebookmap
import armpipelines

# project imports
from codebookmap import BRFSS_MAP
from armpipelines import make_arm_a
from armpipelines import make_arm_b

#pandas and numpy import
import pandas as pd
import numpy as np

# silence warnings
import warnings
warnings.filterwarnings('ignore')

# Pipeline and simple imputer imports
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [2]:
dev = pd.read_parquet("../../data/dev.parquet")
dev.head()

,_STATE,_METSTAT,_URBSTAT,SEXVAR,_AGE80,_RACE,MARITAL,EDUCA,RENTHOM1,VETERAN3,...,HIVTST7,PNEUVAC4,TETANUS1,FRUIT2,FRUITJU2,FVGREEN1,FRENCHF1,POTATOE1,VEGETAB2,CLASS
0,33.0,2.0,2.0,2.0,54.0,1.0,1.0,6.0,1.0,1.0,...,2.0,1.0,1.0,203.0,203.0,203.0,555.0,301.0,202.0,1
1,39.0,1.0,1.0,1.0,59.0,1.0,1.0,4.0,1.0,1.0,...,1.0,1.0,1.0,203.0,204.0,202.0,203.0,306.0,203.0,1
2,19.0,1.0,1.0,1.0,74.0,1.0,1.0,4.0,1.0,1.0,...,2.0,2.0,3.0,310.0,555.0,101.0,300.0,300.0,101.0,1
3,36.0,1.0,1.0,2.0,46.0,2.0,1.0,6.0,1.0,2.0,...,1.0,2.0,4.0,102.0,555.0,101.0,201.0,101.0,101.0,0
4,47.0,2.0,1.0,1.0,63.0,1.0,2.0,5.0,1.0,1.0,...,2.0,2.0,NaN,201.0,201.0,102.0,102.0,555.0,101.0,0


In [3]:
summary_stats = dev['PHYSHLTH'].describe()
desc_with_na = summary_stats.append(pd.Series({
    'NaN_count': dev['PHYSHLTH'].isna().sum()
}))
print(desc_with_na)

count        50000.000000
mean            58.047120
std             37.810851
min              1.000000
25%             14.000000
50%             88.000000
75%             88.000000
max             99.000000
NaN_count        0.000000
dtype: float64


In [4]:
df_arm_b_X, df_arm_b_y = make_arm_b(dev)
df_arm_a_X, df_arm_a_y = make_arm_a(dev)

In [ ]:
#Pipeline A
# Numeric branch (Arm A)
naive_numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean', add_indicator=False)),
    ('scaler', StandardScaler())
])

# Nominal branch (Arm A)
naive_nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# preprocessor (Arm A)
preprocessor_arm_a = ColumnTransformer(
    transformers=[
        ('num', naive_numeric_transformer, numeric_cols),
        ('cat', naive_nominal_transformer, nominal_cols)
    ]
)

In [ ]:
#Pipeline B
#  Rule 1(Arm B) : One-Hot Encode (including 'Unknown')
nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

#  Rule 2 (Arm B): Impute -> Missing Indicators -> Standardize
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler())
])

# preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', nominal_transformer, nominal_cols)
    ]
)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, auc
